# Incremental Sales Data ETL Pipeline
## Raw → Bronze → Silver → Gold (Delta Lake Architecture)

**Features:** Delta Lake with MERGE/UPSERT, Z-Ordering, OPTIMIZE, Caching

## 1. Widget Parameters (Set via ADF Pipeline)

In [0]:
# Create widgets for ADF parameters
dbutils.widgets.text("trigger_date", "", "Trigger Date (YYYY-MM-DD)")
dbutils.widgets.text("trigger_time", "", "Trigger Time (HH:MM:SS)")
dbutils.widgets.text("pipeline_run_id", "", "Pipeline Run ID")
dbutils.widgets.dropdown("processing_mode", "incremental", ["incremental", "full"], "Processing Mode")
dbutils.widgets.text("source_container", "raw", "Source Container")
dbutils.widgets.text("storage_account", "adlsankit1", "Storage Account Name")

In [0]:
# Retrieve widget values
trigger_date = dbutils.widgets.get("trigger_date")
trigger_time = dbutils.widgets.get("trigger_time")
pipeline_run_id = dbutils.widgets.get("pipeline_run_id")
processing_mode = dbutils.widgets.get("processing_mode")
source_container = dbutils.widgets.get("source_container")
storage_account = dbutils.widgets.get("storage_account")

# Set defaults if not provided
from datetime import datetime, timedelta

if not trigger_date:
    trigger_date = datetime.now().strftime("%Y-%m-%d")
if not trigger_time:
    trigger_time = datetime.now().strftime("%H:%M:%S")
if not pipeline_run_id:
    pipeline_run_id = f"manual_run_{datetime.now().strftime('%Y%m%d%H%M%S')}"

print(f"Trigger: {trigger_date} {trigger_time} | Mode: {processing_mode} | Run ID: {pipeline_run_id}")

## 2. Configuration & Setup

In [0]:
from pyspark.sql.functions import (
    col, lit, current_timestamp, to_date, year, month, dayofmonth, 
    sum as spark_sum, count, avg, row_number, when, coalesce, trim, upper,
    sha2, concat_ws, max as spark_max, min as spark_min
)
from pyspark.sql.window import Window
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DoubleType, DateType, TimestampType
from delta.tables import DeltaTable
import time

# Data paths
RAW_PATH = f"/mnt/{source_container}/sales_data"
BRONZE_PATH = "/mnt/raw/bronze/sales"
SILVER_PATH = "/mnt/raw/silver/sales"
GOLD_PATH = "/mnt/gold/sales"
CHECKPOINT_PATH = "/mnt/raw/checkpoints/sales"

print(f"Paths: RAW={RAW_PATH}, BRONZE={BRONZE_PATH}, SILVER={SILVER_PATH}, GOLD={GOLD_PATH}")

## 3. Mount Additional Containers (if not mounted)

In [0]:

def mount_container(container_name, mount_point):
    """Mount ADLS container if not already mounted"""
    configs = {
        f"fs.azure.account.key.{storage_account_name}.blob.core.windows.net": access_key
    }
    
    if any(mount.mountPoint == mount_point for mount in dbutils.fs.mounts()):
        print(f"{mount_point} already mounted")
        return
    
    dbutils.fs.mount(
        source=f"wasbs://{container_name}@{storage_account_name}.blob.core.windows.net/",
        mount_point=mount_point,
        extra_configs=configs
    )
    print(f"Mounted {container_name} to {mount_point}")

# Mount required containers
mount_container("raw", "/mnt/raw")
mount_container("gold", "/mnt/gold")
mount_container("landing", "/mnt/landing")

In [0]:
import random
from datetime import datetime, timedelta
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DoubleType

# Generate sample sales data for testing
def create_sample_sales_data(num_records=50, target_date=None):
    """Create randomized sample sales data for demonstration"""
    
    # If no date is passed, use today
    if not target_date:
        target_date = datetime.now().strftime("%Y-%m-%d")
        
    categories = ["Electronics", "Clothing", "Home & Garden", "Toys", "Sports & Outdoors"]
    regions = ["New York", "Los Angeles", "Chicago", "Houston", "Phoenix", "Seattle", "Miami"]
    statuses = ["Completed", "Completed", "Completed", "Pending", "Shipped", "Cancelled"] 
    
    sample_data = []
    
    for _ in range(num_records):
        # Generate random IDs
        order_id = f"ORD{random.randint(10000, 99999)}"
        product_id = f"PROD{random.randint(1, 50):03d}"
        
        # Inject intentional NULLs (10% chance) for data quality testing
        if random.random() > 0.10:
            customer_id = f"CUST{random.randint(1, 100):03d}"
        else:
            customer_id = None
            
        category = random.choice(categories)
        quantity = random.randint(1, 15)
        unit_price = round(random.uniform(10.0, 2999.99), 2)
        region = random.choice(regions)
        status = random.choice(statuses)
        
        # 80% chance to be today's date (for incremental load testing)
        # 20% chance to be an older date
        if random.random() > 0.20:
            order_date = target_date
        else:
            days_ago = random.randint(1, 10)
            order_date = (datetime.strptime(target_date, "%Y-%m-%d") - timedelta(days=days_ago)).strftime("%Y-%m-%d")
            
        sample_data.append((order_id, customer_id, product_id, category, quantity, unit_price, order_date, region, status))

    # Intentionally append a duplicate of the first record to test deduplication in the Silver layer
    if len(sample_data) > 0:
        sample_data.append(sample_data[0])

    schema = StructType([
        StructField("order_id", StringType(), False),
        StructField("customer_id", StringType(), True),
        StructField("product_id", StringType(), False),
        StructField("category", StringType(), True),
        StructField("quantity", IntegerType(), True),
        StructField("unit_price", DoubleType(), True),
        StructField("order_date", StringType(), True),
        StructField("region", StringType(), True),
        StructField("status", StringType(), True)
    ])
    
    df = spark.createDataFrame(sample_data, schema)
    
    # Save to raw path
    df.write.mode("overwrite").option("header", "true").csv(RAW_PATH)
    print(f"✅ Generated {num_records + 1} randomized sample sales records at {RAW_PATH}")
    return df

# Create 2000 random records (passing the trigger_date from ADF so the incremental logic works perfectly)
sample_df = create_sample_sales_data(num_records=2000, target_date=trigger_date)
display(sample_df)

## 5. Watermark Table for Incremental Loading

In [0]:
# Watermark table for tracking incremental loads
WATERMARK_PATH = "/mnt/raw/metadata/watermark"

def get_last_watermark():
    """Get the last processed timestamp from watermark table"""
    try:
        if DeltaTable.isDeltaTable(spark, WATERMARK_PATH):
            watermark_df = spark.read.format("delta").load(WATERMARK_PATH)
            return watermark_df.select(spark_max("last_processed_timestamp")).collect()[0][0]
        return None
    except:
        return None

def update_watermark(processed_timestamp):
    """Update the watermark after successful processing"""
    watermark_schema = StructType([
        StructField("pipeline_run_id", StringType(), False),
        StructField("last_processed_timestamp", TimestampType(), True),
        StructField("updated_at", TimestampType(), True)
    ])
    watermark_df = spark.createDataFrame(
        [(pipeline_run_id, processed_timestamp, current_timestamp())], 
        watermark_schema
    )
    watermark_df.write.format("delta").mode("append").save(WATERMARK_PATH)

last_watermark = get_last_watermark()
print(f"Last Watermark: {last_watermark} | Current Trigger: {trigger_date}")

## 6. BRONZE LAYER - Raw Ingestion

In [0]:
bronze_start_time = time.time()

# Read raw CSV files
raw_df = spark.read.option("header", "true").option("inferSchema", "true").csv(RAW_PATH)

# Filter for incremental loading
if processing_mode == "incremental" and last_watermark:
    raw_df = raw_df.filter(col("order_date") >= trigger_date)
    print(f"Incremental: Processing data after {last_watermark}")
else:
    print("Full Load: Processing all data")

print(f"Raw records: {raw_df.count()}")
display(raw_df)

In [0]:
# Add ingestion metadata for Bronze layer
bronze_df = raw_df \
    .withColumn("ingestion_timestamp", current_timestamp()) \
    .withColumn("ingestion_date", to_date(current_timestamp())) \
    .withColumn("pipeline_run_id", lit(pipeline_run_id)) \
    .withColumn("source_file", lit(RAW_PATH)) \
    .withColumn("record_hash", sha2(concat_ws("||", *raw_df.columns), 256))

bronze_df.printSchema()

In [0]:
# Write Bronze layer with MERGE/UPSERT logic
if DeltaTable.isDeltaTable(spark, BRONZE_PATH):
    bronze_delta = DeltaTable.forPath(spark, BRONZE_PATH)
    bronze_delta.alias("target").merge(
        bronze_df.alias("source"),
        "target.order_id = source.order_id"
    ).whenMatchedUpdate(
        condition="target.record_hash != source.record_hash",
        set={col: f"source.{col}" for col in bronze_df.columns}
    ).whenNotMatchedInsertAll().execute()
else:
    bronze_df.write.format("delta").mode("overwrite").partitionBy("ingestion_date").save(BRONZE_PATH)

bronze_duration = time.time() - bronze_start_time
print(f"Bronze Layer: {bronze_duration:.2f}s | Records: {spark.read.format('delta').load(BRONZE_PATH).count()}")

## 7. SILVER LAYER - Cleansed & Standardized

In [0]:
silver_start_time = time.time()

# Read Bronze data (filter for incremental if needed)
bronze_source_df = spark.read.format("delta").load(BRONZE_PATH)
if processing_mode == "incremental":
    bronze_source_df = bronze_source_df.filter(col("ingestion_date") == trigger_date)

print(f"Bronze records for Silver: {bronze_source_df.count()}")

In [0]:
# Data Quality & Cleansing for Silver Layer
silver_df = bronze_source_df \
    .withColumn("customer_id", coalesce(col("customer_id"), lit("UNKNOWN"))) \
    .withColumn("category", coalesce(col("category"), lit("Uncategorized"))) \
    .withColumn("region", coalesce(col("region"), lit("Unknown"))) \
    .withColumn("status", coalesce(col("status"), lit("Unknown"))) \
    .withColumn("quantity", col("quantity").cast(IntegerType())) \
    .withColumn("unit_price", col("unit_price").cast(DoubleType())) \
    .withColumn("order_date", to_date(col("order_date"))) \
    .withColumn("category", upper(trim(col("category")))) \
    .withColumn("region", upper(trim(col("region")))) \
    .withColumn("status", upper(trim(col("status")))) \
    .withColumn("total_amount", col("quantity") * col("unit_price")) \
    .withColumn("order_year", year(col("order_date"))) \
    .withColumn("order_month", month(col("order_date"))) \
    .withColumn("order_day", dayofmonth(col("order_date")))

# Deduplicate (keep latest per order_id)
window_spec = Window.partitionBy("order_id").orderBy(col("ingestion_timestamp").desc())
silver_df = silver_df \
    .withColumn("row_num", row_number().over(window_spec)) \
    .filter(col("row_num") == 1).drop("row_num") \
    .filter((col("quantity") > 0) & (col("unit_price") > 0)) \
    .withColumn("silver_processed_timestamp", current_timestamp()) \
    .withColumn("is_valid", lit(True))

silver_df.printSchema()

In [0]:
# Write Silver layer with MERGE/UPSERT
if DeltaTable.isDeltaTable(spark, SILVER_PATH):
    silver_delta = DeltaTable.forPath(spark, SILVER_PATH)
    silver_delta.alias("target").merge(
        silver_df.alias("source"),
        "target.order_id = source.order_id"
    ).whenMatchedUpdate(
        condition="target.record_hash != source.record_hash",
        set={col: f"source.{col}" for col in silver_df.columns}
    ).whenNotMatchedInsertAll().execute()
else:
    silver_df.write.format("delta").mode("overwrite").partitionBy("order_year", "order_month").save(SILVER_PATH)

silver_duration = time.time() - silver_start_time
print(f"Silver Layer: {silver_duration:.2f}s | Records: {spark.read.format('delta').load(SILVER_PATH).count()}")

## 8. GOLD LAYER - Business KPIs & Aggregations

In [0]:
gold_start_time = time.time()
silver_source_df = spark.read.format("delta").load(SILVER_PATH)
print(f"Silver records for Gold: {silver_source_df.count()}")

In [0]:
# Gold: Sales by Region
gold_region_df = silver_source_df.groupBy("region", "order_year", "order_month").agg(
    count("order_id").alias("total_orders"),
    spark_sum("quantity").alias("total_quantity"),
    spark_sum("total_amount").alias("total_revenue"),
    avg("total_amount").alias("avg_order_value")
).withColumn("gold_processed_timestamp", current_timestamp())

GOLD_REGION_PATH = f"{GOLD_PATH}/sales_by_region"
gold_region_df.write.format("delta").mode("overwrite").partitionBy("order_year", "order_month").save(GOLD_REGION_PATH)
display(gold_region_df)

In [0]:
# Gold: Sales by Category
gold_category_df = silver_source_df.groupBy("category", "order_year", "order_month").agg(
    count("order_id").alias("total_orders"),
    spark_sum("quantity").alias("total_quantity"),
    spark_sum("total_amount").alias("total_revenue"),
    count(when(col("status") == "COMPLETED", 1)).alias("completed_orders"),
    count(when(col("status") == "PENDING", 1)).alias("pending_orders")
).withColumn("completion_rate", (col("completed_orders") / col("total_orders") * 100).cast(DoubleType())) \
 .withColumn("gold_processed_timestamp", current_timestamp())

GOLD_CATEGORY_PATH = f"{GOLD_PATH}/sales_by_category"
gold_category_df.write.format("delta").mode("overwrite").partitionBy("order_year", "order_month").save(GOLD_CATEGORY_PATH)
display(gold_category_df)

In [0]:
# Gold: Customer Analytics
gold_customer_df = silver_source_df.groupBy("customer_id").agg(
    count("order_id").alias("total_orders"),
    spark_sum("total_amount").alias("lifetime_value"),
    avg("total_amount").alias("avg_order_value"),
    spark_max("order_date").alias("last_order_date"),
    spark_min("order_date").alias("first_order_date")
).withColumn("customer_segment", 
    when(col("lifetime_value") > 5000, "PLATINUM")
    .when(col("lifetime_value") > 2000, "GOLD")
    .when(col("lifetime_value") > 500, "SILVER")
    .otherwise("BRONZE")
).withColumn("gold_processed_timestamp", current_timestamp())

GOLD_CUSTOMER_PATH = f"{GOLD_PATH}/customer_analytics"
gold_customer_df.write.format("delta").mode("overwrite").save(GOLD_CUSTOMER_PATH)
display(gold_customer_df)

In [0]:
# Gold: Daily Sales Dashboard
gold_daily_df = silver_source_df.groupBy("order_date", "region", "category").agg(
    count("order_id").alias("daily_orders"),
    spark_sum("total_amount").alias("daily_revenue"),
    spark_sum("quantity").alias("daily_quantity")
).withColumn("gold_processed_timestamp", current_timestamp())

GOLD_DAILY_PATH = f"{GOLD_PATH}/daily_sales_dashboard"
gold_daily_df.write.format("delta").mode("overwrite").partitionBy("order_date").save(GOLD_DAILY_PATH)

gold_duration = time.time() - gold_start_time
print(f"Gold Layer: {gold_duration:.2f}s")
display(gold_daily_df)

## 10. PERFORMANCE OPTIMIZATION

In [0]:
# Baseline query performance measurement
before_opt_start = time.time()

sample_query_df = spark.read.format("delta").load(SILVER_PATH) \
    .filter(col("region") == "NEW YORK") \
    .filter(col("category") == "ELECTRONICS") \
    .select("order_id", "customer_id", "total_amount")
sample_query_df.collect()

before_optimization_runtime = time.time() - before_opt_start
print(f"Before Optimization: {before_optimization_runtime:.4f}s")

In [0]:
# Run OPTIMIZE with Z-Ordering on Silver table
optimize_start = time.time()
spark.sql(f"OPTIMIZE delta.`{SILVER_PATH}` ZORDER BY (region, category, order_date)")
print(f"OPTIMIZE + Z-Order completed in {time.time() - optimize_start:.2f}s")

In [0]:
# Cache frequently accessed data
silver_cached_df = spark.read.format("delta").load(SILVER_PATH).cache()
silver_cached_df.count()  # Trigger cache
print("Silver table cached")

In [0]:
# Query performance after optimization
after_opt_start = time.time()

optimized_query_df = silver_cached_df \
    .filter(col("region") == "NEW YORK") \
    .filter(col("category") == "ELECTRONICS") \
    .select("order_id", "customer_id", "total_amount")
optimized_query_df.collect()

after_optimization_runtime = time.time() - after_opt_start
improvement = ((before_optimization_runtime - after_optimization_runtime) / before_optimization_runtime) * 100 if before_optimization_runtime > 0 else 0
print(f"After Optimization: {after_optimization_runtime:.4f}s | Improvement: {improvement:.1f}%")

In [0]:
# Performance Summary
bronze_count = spark.read.format("delta").load(BRONZE_PATH).count()
silver_count = spark.read.format("delta").load(SILVER_PATH).count()

print(f"""
Performance Report
------------------
Layer Times: Bronze={bronze_duration:.2f}s, Silver={silver_duration:.2f}s, Gold={gold_duration:.2f}s
Query Optimization: {before_optimization_runtime:.4f}s -> {after_optimization_runtime:.4f}s ({improvement:.1f}% faster)
Records: Bronze={bronze_count}, Silver={silver_count}, Cleaned={bronze_count - silver_count}
""")

In [0]:
# Update watermark after successful processing
# processing_timestamp = datetime.strptime(f"{trigger_date} {trigger_time}", "%Y-%m-%d %H:%M:%S")
# update_watermark(processing_timestamp)
# print(f"Watermark updated: {processing_timestamp}")

In [0]:
# Pipeline Summary
print(f"""
Pipeline Complete
-----------------
Run ID: {pipeline_run_id}
Date: {trigger_date} | Mode: {processing_mode}
Gold Tables: {GOLD_PATH}/[sales_by_region, sales_by_category, customer_analytics, daily_sales_dashboard]
""")

In [0]:
# Return success status for ADF
dbutils.notebook.exit("SUCCESS")